# Process Files from CSV Mapping

> **Note:** Notebooks in this directory use CLI commands rather than Python library imports.

This notebook loads a filename mapping CSV (from `renaming_file_template`), downloads files from S3, converts to COG, and uploads with new names.

## Workflow
1. Configure settings (bucket, event, CSV path, compression)
2. Load CSV mapping with `pd.read_csv()`
3. Preview category breakdown and sample transformations
4. Optionally filter by category, size, etc.
5. Process each file: download -> check CRS -> reproject if needed -> convert to COG -> validate -> upload
6. Results summary with optional CSV save

---

## Step 1: Configuration

In [ ]:
import pandas as pd
import subprocess
import json
import time
import os
import re
from pathlib import Path
from datetime import datetime

# ========================================
# S3 Configuration
# ========================================
BUCKET = 'nasa-disasters'  # S3 bucket (DO NOT CHANGE)

# ========================================
# Event Details
# ========================================
EVENT_NAME = '202510_Flood_AK'
SOURCE = "TBD"  # Data origin (e.g., USGS, Copernicus, CSDA, TBD)
SUB_PRODUCT_NAME = 'sentinel2'

# ========================================
# CSV Mapping Configuration
# ========================================
CSV_DIR = f'file-mapping/{EVENT_NAME}'
CSV_FILENAME = f'{EVENT_NAME}-{SUB_PRODUCT_NAME}.csv'

# ========================================
# COG Compression Settings
# ========================================
COMPRESSION = 'deflate'     # Compression type: deflate, lzw, zstd, etc.
COMPRESSION_LEVEL = 9       # Compression level (1-9 for deflate/zstd)
BLOCKSIZE = 512             # Tile size for COG
OVERVIEW_LEVELS = 5         # Number of overview levels
OVERVIEW_RESAMPLING = 'nearest'  # Overview resampling method

# ========================================
# Processing Options
# ========================================
OVERWRITE = True            # Replace existing files in S3
# Target CRS for the output COG.
# None preserves the source projection (fastest, no warp).
# Uncomment the EPSG:3857 line if you plan to push the COG through
# veda-data-airflow's build_stac, which trips on the EPSG:4326 ensemble.
TARGET_CRS = None
# TARGET_CRS = "EPSG:3857"
SAVE_RESULTS = True         # Save processing results to CSV
OUTPUT_DIR = 'csv_stats'    # Directory for results CSV

# ========================================
# Temp directory
# ========================================
TMP_DIR = '/tmp'

print(f"Event:       {EVENT_NAME}")
print(f"Bucket:      {BUCKET}")
print(f"Compression: {COMPRESSION} (level {COMPRESSION_LEVEL})")
print(f"Overwrite:   {OVERWRITE}")
print(f"Target CRS:  {TARGET_CRS}")

In [ ]:
from shared_utils import PROCESSOR_STRING
ACTIVATION_METADATA = {
    "ACTIVATION_EVENT": EVENT_NAME,
    "SOURCE": SOURCE,
    "PROCESSOR": PROCESSOR_STRING,
    # Add any custom key-value pairs here
}


## Step 2: Load CSV Mapping

In [ ]:
csv_path = Path(CSV_DIR) / CSV_FILENAME

print("LOADING CSV MAPPING")
print("=" * 80)
print(f"\nLooking for: {csv_path}")

if not csv_path.exists():
    print(f"\nERROR: CSV file not found!")
    print(f"Make sure you've run 'renaming_file_template' first to create the mapping CSV.")
    print(f"Expected location: {csv_path.absolute()}")
    mapping_df = pd.DataFrame()
else:
    mapping_df = pd.read_csv(csv_path)
    print(f"\nSuccessfully loaded CSV")
    print(f"\nMapping details:")
    print(f"   Total entries: {len(mapping_df)}")
    print(f"\nRequired columns: {list(mapping_df.columns)}")
    
    # Verify required columns exist
    required_cols = ['original_s3_path', 'output_s3_path', 'nodata_value']
    missing = [c for c in required_cols if c not in mapping_df.columns]
    if missing:
        print(f"\nWARNING: Missing required columns: {missing}")
    
    display(mapping_df.head(10))

## Step 3: Preview

In [ ]:
if not mapping_df.empty:
    print("CATEGORY BREAKDOWN")
    print("=" * 80)
    
    if 'category' in mapping_df.columns:
        category_stats = mapping_df.groupby('category').agg(
            count=('category', 'size'),
            total_size_gb=('file_size_gb', 'sum') if 'file_size_gb' in mapping_df.columns else ('category', 'size')
        ).reset_index()
        print(f"\n{category_stats.to_string(index=False)}")
    
    print(f"\n\nSAMPLE TRANSFORMATIONS")
    print("=" * 80)
    sample = mapping_df.head(3)
    for _, row in sample.iterrows():
        print(f"\n  Source: s3://{BUCKET}/{row['original_s3_path']}")
        print(f"  Dest:   s3://{BUCKET}/{row['output_s3_path']}")
        if 'nodata_value' in mapping_df.columns:
            print(f"  Nodata: {row['nodata_value']}")
else:
    print("No mapping data loaded. Check Step 2.")

## Step 4: Optional Filtering

In [ ]:
if not mapping_df.empty:
    files_to_process = mapping_df.copy()
    
    # Filter by status if column exists
    if 'status' in files_to_process.columns:
        files_to_process = files_to_process[files_to_process['status'] == 'valid'].copy()
    
    # Optional: Filter by category
    # Uncomment and modify to process only specific categories:
    # CATEGORIES_TO_PROCESS = ['trueColor', 'colorInfrared']
    # files_to_process = files_to_process[files_to_process['category'].isin(CATEGORIES_TO_PROCESS)]
    
    # Optional: Filter by file size
    # MAX_SIZE_GB = 5.0
    # files_to_process = files_to_process[files_to_process['file_size_gb'] <= MAX_SIZE_GB]
    
    print("FILES TO PROCESS")
    print("=" * 80)
    print(f"\nTotal files: {len(files_to_process)}")
    
    if 'file_size_gb' in files_to_process.columns:
        print(f"Total size:  {files_to_process['file_size_gb'].sum():.2f} GB")
    
    if 'category' in files_to_process.columns and len(files_to_process) > 0:
        print(f"\nBy category:")
        for cat, count in files_to_process['category'].value_counts().items():
            print(f"   {cat}: {count} files")
    
    print(f"\nReady to process {len(files_to_process)} files")
else:
    print("No mapping data loaded.")
    files_to_process = pd.DataFrame()

## Step 5: Process Files

For each row: download from S3, check CRS, reproject if needed, convert to COG, validate, upload.

In [ ]:
def get_file_info(filepath):
    """Get file metadata using gdalinfo -json."""
    result = subprocess.run(
        ['gdalinfo', '-json', filepath],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"    WARNING: gdalinfo failed: {result.stderr.strip()}")
        return None
    return json.loads(result.stdout)


def get_crs(info):
    """Extract CRS authority code from gdalinfo output."""
    try:
        crs_wkt = info.get('coordinateSystem', {}).get('wkt', '')
        # Try to find EPSG code in the WKT
        match = re.search(r'AUTHORITY\["EPSG","(\d+)"\]', crs_wkt)
        if match:
            return f"EPSG:{match.group(1)}"
        # Also check ID field in newer WKT2 format
        match = re.search(r'ID\["EPSG",(\d+)\]', crs_wkt)
        if match:
            return f"EPSG:{match.group(1)}"
    except Exception:
        pass
    return None


def s3_file_exists(bucket, key):
    """Check if a file exists in S3 using aws cli."""
    result = subprocess.run(
        ['aws', 's3api', 'head-object', '--bucket', bucket, '--key', key],
        capture_output=True, text=True
    )
    return result.returncode == 0


def cleanup_tmp_files(*filepaths):
    """Remove temporary files."""
    for fp in filepaths:
        if fp and os.path.exists(fp):
            os.remove(fp)


def process_file(row, idx, total):
    """Process a single file: download, check CRS, reproject, convert to COG, validate, upload."""
    source_key = row['original_s3_path']
    dest_key = row['output_s3_path']
    category = row.get('category', 'unknown')
    nodata = row.get('nodata_value', None)
    if pd.isna(nodata):
        nodata = None
    
    src_filename = os.path.basename(source_key)
    dst_filename = os.path.basename(dest_key)
    
    print(f"\n[{idx}/{total}] {src_filename}")
    print(f"    Category: {category}")
    if 'file_size_gb' in row.index:
        print(f"    Size:     {row['file_size_gb']:.2f} GB")
    print(f"    Nodata:   {nodata}")
    print(f"    Output:   {dst_filename}")
    
    # Define temp file paths
    local_src = os.path.join(TMP_DIR, f'src_{src_filename}')
    local_reprojected = os.path.join(TMP_DIR, f'reproj_{dst_filename}')
    local_cog = os.path.join(TMP_DIR, f'cog_{dst_filename}')
    
    try:
        # Check if destination already exists
        if not OVERWRITE and s3_file_exists(BUCKET, dest_key):
            print(f"    SKIPPED (already exists)")
            return 'skipped', 'File already exists'
        
        # --- Download from S3 ---
        print(f"    Downloading from S3...")
        dl_result = subprocess.run(
            ['aws', 's3', 'cp', f's3://{BUCKET}/{source_key}', local_src],
            capture_output=True, text=True
        )
        if dl_result.returncode != 0:
            print(f"    FAILED download: {dl_result.stderr.strip()}")
            return 'failed', f'Download failed: {dl_result.stderr.strip()}'
        
        # --- Check CRS with gdalinfo ---
        print(f"    Checking CRS...")
        info = get_file_info(local_src)
        if info is None:
            cleanup_tmp_files(local_src)
            return 'failed', 'gdalinfo failed'
        
        src_crs = get_crs(info)
        print(f"    Source CRS: {src_crs}")
        
        # --- Reproject if needed ---
        input_for_cog = local_src
        if TARGET_CRS and src_crs and src_crs != TARGET_CRS:
            print(f"    Reprojecting {src_crs} -> {TARGET_CRS}...")
            warp_cmd = [
                'rio', 'warp', local_src, local_reprojected,
                '--dst-crs', TARGET_CRS,
                '--resampling', 'nearest'
            ]
            warp_result = subprocess.run(warp_cmd, capture_output=True, text=True)
            if warp_result.returncode != 0:
                print(f"    FAILED reprojection: {warp_result.stderr.strip()}")
                cleanup_tmp_files(local_src)
                return 'failed', f'Reprojection failed: {warp_result.stderr.strip()}'
            input_for_cog = local_reprojected
        else:
            print(f"    No reprojection needed.")
        
        # --- Convert to COG ---
        print(f"    Converting to COG...")
        cog_cmd = [
            'rio', 'cogeo', 'create',
            input_for_cog, local_cog,
            '--co', f'COMPRESS={COMPRESSION}',
            '--co', f'BLOCKXSIZE={BLOCKSIZE}',
            '--co', f'BLOCKYSIZE={BLOCKSIZE}',
            '--overview-level', str(OVERVIEW_LEVELS),
            '--overview-resampling', OVERVIEW_RESAMPLING,
        ]
        if nodata is not None:
            cog_cmd.extend(['--nodata', str(nodata)])
        
        cog_result = subprocess.run(cog_cmd, capture_output=True, text=True)
        if cog_result.returncode != 0:
            print(f"    FAILED COG conversion: {cog_result.stderr.strip()}")
            cleanup_tmp_files(local_src, local_reprojected)
            return 'failed', f'COG conversion failed: {cog_result.stderr.strip()}'
        
        # --- Validate COG ---
        print(f"    Validating COG...")
        val_result = subprocess.run(
            ['rio', 'cogeo', 'validate', local_cog],
            capture_output=True, text=True
        )
        if val_result.returncode != 0 or 'is NOT a valid' in val_result.stdout:
            print(f"    WARNING: COG validation issue: {val_result.stdout.strip()}")
        else:
            print(f"    COG validation passed.")
        
        # --- Upload to S3 ---
        print(f"    Uploading to S3...")
        ul_result = subprocess.run(
            ['aws', 's3', 'cp', local_cog, f's3://{BUCKET}/{dest_key}'],
            capture_output=True, text=True
        )
        if ul_result.returncode != 0:
            print(f"    FAILED upload: {ul_result.stderr.strip()}")
            cleanup_tmp_files(local_src, local_reprojected, local_cog)
            return 'failed', f'Upload failed: {ul_result.stderr.strip()}'
        
        # --- Cleanup ---
        cleanup_tmp_files(local_src, local_reprojected, local_cog)
        print(f"    SUCCESS")
        return 'success', None
    
    except Exception as e:
        cleanup_tmp_files(local_src, local_reprojected, local_cog)
        print(f"    ERROR: {str(e)}")
        return 'failed', str(e)


print("Processing functions defined.")

In [ ]:
if not files_to_process.empty:
    print("STARTING COG PROCESSING")
    print("=" * 80)
    print(f"\nProcessing {len(files_to_process)} files.")
    print(f"Compression: {COMPRESSION} (level {COMPRESSION_LEVEL})")
    print(f"Overwrite:   {OVERWRITE}")
    print(f"Target CRS:  {TARGET_CRS}")
    
    results = []
    total = len(files_to_process)
    
    for i, (idx, row) in enumerate(files_to_process.iterrows(), 1):
        start_time = time.time()
        
        status, error = process_file(row, i, total)
        elapsed = time.time() - start_time
        
        results.append({
            'source_file': os.path.basename(row['original_s3_path']),
            'output_file': os.path.basename(row['output_s3_path']),
            'category': row.get('category', 'unknown'),
            'status': status,
            'time_seconds': round(elapsed, 1),
            'error': error
        })
        
        if status == 'success':
            print(f"    ({elapsed:.1f}s)")
    
    results_df = pd.DataFrame(results)
    print("\n" + "=" * 80)
    print("PROCESSING COMPLETE")
else:
    print("No files to process.")
    results_df = pd.DataFrame()

## Step 6: Results Summary

In [ ]:
if not results_df.empty:
    print("RESULTS SUMMARY")
    print("=" * 80)
    
    total = len(results_df)
    success_count = len(results_df[results_df['status'] == 'success'])
    failed_count = len(results_df[results_df['status'] == 'failed'])
    skipped_count = len(results_df[results_df['status'] == 'skipped'])
    
    print(f"\n  Total processed: {total}")
    print(f"  Successful:      {success_count}")
    print(f"  Failed:          {failed_count}")
    print(f"  Skipped:         {skipped_count}")
    
    if 'time_seconds' in results_df.columns:
        total_time = results_df['time_seconds'].sum()
        print(f"\n  Total time: {total_time:.1f}s ({total_time/60:.1f} min)")
        if success_count > 0:
            avg_time = results_df[results_df['status'] == 'success']['time_seconds'].mean()
            print(f"  Avg time per file: {avg_time:.1f}s")
    
    if failed_count > 0:
        print(f"\n  FAILED FILES:")
        failed = results_df[results_df['status'] == 'failed']
        for _, row in failed.iterrows():
            print(f"    - {row['source_file']}: {row['error']}")
    
    # Save results to CSV
    if SAVE_RESULTS:
        output_path = Path(OUTPUT_DIR) / EVENT_NAME
        output_path.mkdir(parents=True, exist_ok=True)
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        results_path = output_path / f'processing_results_{timestamp}.csv'
        results_df.to_csv(results_path, index=False)
        print(f"\n  Results saved to: {results_path.absolute()}")

else:
    print("No results to display.")